In [ ]:
# Bernoulli Naive Bayes on Binned / One-Hot Features

import pandas as pd

from pathlib import Path
import json
import joblib
import warnings

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import BernoulliNB
from sklearn.inspection import permutation_importance

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

warnings.filterwarnings("ignore")

In [ ]:
# Helper Functions

def ks_statistic(y_true, y_score):
    temp = pd.DataFrame({
        "y_true": y_true,
        "y_score": y_score
    }).sort_values("y_score", ascending=False)

    temp["good"] = (temp["y_true"] == 0).astype(int)
    temp["bad"] = (temp["y_true"] == 1).astype(int)

    temp["cum_good"] = temp["good"].cumsum() / temp["good"].sum()
    temp["cum_bad"] = temp["bad"].cumsum() / temp["bad"].sum()

    return (temp["cum_bad"] - temp["cum_good"]).abs().max()


def calculate_classification_diagnostics(
    y_train,
    p_train,
    y_val,
    p_val,
    threshold=0.50
):
    rows = []

    for dataset_name, y_true, p_score in [
        ("train", y_train, p_train),
        ("validation", y_val, p_val)
    ]:

        y_pred = (p_score >= threshold).astype(int)

        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

        rows.append({
            "dataset": dataset_name,
            "threshold": threshold,
            "auc": roc_auc_score(y_true, p_score),
            "gini": 2 * roc_auc_score(y_true, p_score) - 1,
            "ks": ks_statistic(y_true, p_score),
            "pr_auc": average_precision_score(y_true, p_score),
            "log_loss": log_loss(y_true, p_score),
            "brier_score": brier_score_loss(y_true, p_score),
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "true_negative": tn,
            "false_positive": fp,
            "false_negative": fn,
            "true_positive": tp
        })

    return pd.DataFrame(rows)

print("Diagnostic helpers loaded.")


def get_next_model_number(model_registry, prefix):
    existing_numbers = []

    for model_num in model_registry.keys():
        if model_num.startswith(prefix):
            number_part = model_num.replace(prefix, "")
            if number_part.isdigit():
                existing_numbers.append(int(number_part))

    return max(existing_numbers) + 1 if existing_numbers else 1


def run_nb_model(
    model_registry,
    model,
    model_number,
    model_name,
    X_train,
    X_val,
    y_train,
    y_val,
    features,
    threshold=0.50,
    analyst_comments="",
    run_permutation=True,
    permutation_scoring="roc_auc",
    permutation_repeats=5,
    display_outputs=True
):

    if model_number in model_registry:
        raise ValueError(f"{model_number} already exists.")

    features = list(features)

    model.fit(X_train[features], y_train)

    p_train = model.predict_proba(X_train[features])[:, 1]
    p_val = model.predict_proba(X_val[features])[:, 1]

    diagnostics_table = calculate_classification_diagnostics(
        y_train=y_train,
        p_train=p_train,
        y_val=y_val,
        p_val=p_val,
        threshold=threshold
    )

    if run_permutation:
        perm_result = permutation_importance(
            model,
            X_val[features],
            y_val,
            scoring=permutation_scoring,
            n_repeats=permutation_repeats,
            random_state=42,
            n_jobs=-1
        )

        permutation_importance_df = pd.DataFrame({
            "variable": features,
            "permutation_importance_mean": perm_result.importances_mean,
            "permutation_importance_std": perm_result.importances_std
        }).sort_values(
            "permutation_importance_mean",
            ascending=False
        ).reset_index(drop=True)
    else:
        permutation_importance_df = pd.DataFrame()

    metadata = {
        "model_number": model_number,
        "model_name": model_name,
        "model_class": type(model).__name__,
        "features": features,
        "num_features": len(features),
        "threshold": threshold,
        "parameters": model.get_params(),
        "analyst_comments": analyst_comments
    }

    model_registry[model_number] = {
        "metadata": metadata,
        "model": model,
        "diagnostics": diagnostics_table,
        "feature_importance": pd.DataFrame(),
        "permutation_importance": permutation_importance_df,
        "p_train": p_train,
        "p_val": p_val
    }

    if display_outputs:
        print("=" * 100)
        print(f"{model_number}: {model_name}")
        print("=" * 100)

        print("\nDIAGNOSTICS:")
        display(diagnostics_table)

        print("\nPERMUTATION IMPORTANCE:")
        display(permutation_importance_df.head(15))

        if analyst_comments:
            print("\nANALYST COMMENTS:")
            print(analyst_comments)

    return model_registry

def save_model_artifact(model_registry, model_id, model_dir, config_dir):
    record = model_registry[model_id]

    model_path = model_dir / f"{model_id}.joblib"
    config_path = config_dir / f"{model_id}_config.json"

    joblib.dump(record["model"], model_path)

    metadata = record.get("metadata", {}).copy()

    config = {
        "model_id": model_id,
        "model_class": str(type(record["model"]).__name__),
        "metadata": metadata,
        "features": metadata.get("features", None),
        "num_features": metadata.get("num_features", None),
    }

    if "diagnostics" in record:
        try:
            config["diagnostics_preview"] = record["diagnostics"].to_dict(orient="records")
        except Exception:
            pass

    with open(config_path, "w") as f:
        json.dump(config, f, indent=4, default=str)

Diagnostic helpers loaded.


In [2]:
# Paths

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

MODEL_DIR = OUTPUT_DIR / "saved_models"
CONFIG_DIR = OUTPUT_DIR / "model_configs"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

print("Project Root:", PROJECT_ROOT)
print("Data Directory:", DATA_DIR)
print("Output Directory:", OUTPUT_DIR)

Project Root: /Users/sumanchattopadhyay/Documents/Documents - Suman’s MacBook Air/Projects/credit-risk-mlops-lab
Data Directory: /Users/sumanchattopadhyay/Documents/Documents - Suman’s MacBook Air/Projects/credit-risk-mlops-lab/data
Output Directory: /Users/sumanchattopadhyay/Documents/Documents - Suman’s MacBook Air/Projects/credit-risk-mlops-lab/outputs


In [3]:
# Load Binned Model-Ready Dataset

df_bins = pd.read_parquet(DATA_DIR / "df_logit_bins_ready.parquet")

print("Binned dataset shape:", df_bins.shape)
display(df_bins.head())

Binned dataset shape: (149390, 12)


,SeriousDlqin2yrs,RevolvingUtilization_final_bin,age_final_bin_v2,MonthlyIncome_final_bin,MonthlyIncome_missing_flag_bin,DebtRatio_high_flag_bin,NumberOfDependents_final_bin,NumberOfDependents_missing_flag_bin,RealEstateLoans_final_bin,NumberOfTime30-59DaysPastDueNotWorse_final_bin,NumberOfTime60-89DaysPastDueNotWorse_final_bin,NumberOfTimes90DaysLate_final_bin
0,1,75-100%,40-49,5000-10000,0,0,2,0,3+,2,0,0
1,0,75-100%,40-49,0-5000,0,0,1,0,0,0,0,0
2,0,50-75%,21-39,0-5000,0,0,0,0,0,1,0,1
3,0,15-30%,21-39,0-5000,0,0,0,0,0,0,0,0
4,0,75-100%,40-49,10000+,0,0,0,0,1-2,1,0,0


In [4]:
# Define Target + Candidate Features

target = "SeriousDlqin2yrs"

candidate_features = [
    col for col in df_bins.columns
    if col != target
]

print("Target:", target)
print("Number of candidate features:", len(candidate_features))
print(candidate_features)

Target: SeriousDlqin2yrs
Number of candidate features: 11
['RevolvingUtilization_final_bin', 'age_final_bin_v2', 'MonthlyIncome_final_bin', 'MonthlyIncome_missing_flag_bin', 'DebtRatio_high_flag_bin', 'NumberOfDependents_final_bin', 'NumberOfDependents_missing_flag_bin', 'RealEstateLoans_final_bin', 'NumberOfTime30-59DaysPastDueNotWorse_final_bin', 'NumberOfTime60-89DaysPastDueNotWorse_final_bin', 'NumberOfTimes90DaysLate_final_bin']


In [21]:
# One-Hot Encode Binned Features for BernoulliNB

X_bnb = pd.get_dummies(
    df_bins[candidate_features],
    drop_first=False,
    dtype=int
)

y_bnb = df_bins[target].copy()

print("X_bnb shape:", X_bnb.shape)
print("y_bnb shape:", y_bnb.shape)

display(X_bnb.head())

X_bnb shape: (149390, 41)
y_bnb shape: (149390,)


,RevolvingUtilization_final_bin_=0,RevolvingUtilization_final_bin_0-5%,RevolvingUtilization_final_bin_5-15%,RevolvingUtilization_final_bin_15-30%,RevolvingUtilization_final_bin_30-50%,RevolvingUtilization_final_bin_50-75%,RevolvingUtilization_final_bin_75-100%,RevolvingUtilization_final_bin_100%+,age_final_bin_v2_21-39,age_final_bin_v2_40-49,...,NumberOfTime30-59DaysPastDueNotWorse_final_bin_2,NumberOfTime30-59DaysPastDueNotWorse_final_bin_3+,NumberOfTime60-89DaysPastDueNotWorse_final_bin_0,NumberOfTime60-89DaysPastDueNotWorse_final_bin_1,NumberOfTime60-89DaysPastDueNotWorse_final_bin_2,NumberOfTime60-89DaysPastDueNotWorse_final_bin_3+,NumberOfTimes90DaysLate_final_bin_0,NumberOfTimes90DaysLate_final_bin_1,NumberOfTimes90DaysLate_final_bin_2,NumberOfTimes90DaysLate_final_bin_3+
0,0,0,0,0,0,0,1,0,0,1,...,1,0,1,0,0,0,1,0,0,0
1,0,0,0,0,0,0,1,0,0,1,...,0,0,1,0,0,0,1,0,0,0
2,0,0,0,0,0,1,0,0,1,0,...,0,0,1,0,0,0,0,1,0,0
3,0,0,0,1,0,0,0,0,1,0,...,0,0,1,0,0,0,1,0,0,0
4,0,0,0,0,0,0,1,0,0,1,...,0,0,1,0,0,0,1,0,0,0


In [22]:
# Train/validation split
X_train, X_val, y_train, y_val = train_test_split(
    X_bnb,
    y_bnb,
    test_size=0.30,
    stratify=y_bnb,
    random_state=42
)

candidate_features_bnb = X_bnb.columns.tolist()

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("Train bad rate:", y_train.mean())
print("Val bad rate  :", y_val.mean())

X_train: (104573, 41)
X_val  : (44817, 41)
Train bad rate: 0.06699626098514913
Val bad rate  : 0.06700582368297744


In [23]:
# Initiate registry and model

bnb_registry = {}

bnb_model = BernoulliNB()

In [24]:
# BNB001 - Bernoulli Naive Bayes Baseline
# All Binned / One-Hot Features

bnb_registry = run_nb_model(
    model_registry=bnb_registry,
    model=bnb_model,
    model_number="BNB001",
    model_name="Bernoulli Naive Bayes - All Binned Features",
    X_train=X_train,
    X_val=X_val,
    y_train=y_train,
    y_val=y_val,
    features=candidate_features_bnb,
    analyst_comments=(
        "Baseline Bernoulli Naive Bayes using all one-hot encoded "
        "binned features. BernoulliNB is naturally suited for binary "
        "indicator inputs."
    ),
    run_permutation=True,
    permutation_repeats=5,
    display_outputs=True
)

BNB001: Bernoulli Naive Bayes - All Binned Features

DIAGNOSTICS:


,dataset,threshold,auc,gini,ks,pr_auc,log_loss,brier_score,accuracy,precision,recall,f1,true_negative,false_positive,false_negative,true_positive
0,train,0.5,0.849783,0.699566,0.543040,0.366881,0.348811,0.076180,0.908954,0.369730,0.509420,0.428477,91483,6084,3437,3569
1,validation,0.5,0.855490,0.710981,0.558075,0.378108,0.343121,0.075482,0.909789,0.374031,0.514153,0.433039,39230,2584,1459,1544



PERMUTATION IMPORTANCE:


,variable,permutation_importance_mean,permutation_importance_std
0,RevolvingUtilization_final_bin_0-5%,0.015837,0.000812
1,RevolvingUtilization_final_bin_75-100%,0.012093,0.000767
2,NumberOfTime30-59DaysPastDueNotWorse_final_bin_0,0.011701,0.000295
3,NumberOfTimes90DaysLate_final_bin_0,0.007398,0.000677
4,RevolvingUtilization_final_bin_5-15%,0.006492,0.000640
5,NumberOfTime30-59DaysPastDueNotWorse_final_bin_3+,0.004610,0.000486
6,RevolvingUtilization_final_bin_100%+,0.004105,0.000285
7,NumberOfTime60-89DaysPastDueNotWorse_final_bin_0,0.003794,0.000620
8,NumberOfTime30-59DaysPastDueNotWorse_final_bin_2,0.002821,0.000239
9,age_final_bin_v2_70+,0.002671,0.000225



ANALYST COMMENTS:
Baseline Bernoulli Naive Bayes using all one-hot encoded binned features. BernoulliNB is naturally suited for binary indicator inputs.


## Why Bernoulli Naive Bayes Materially Outperformed Gaussian Naive Bayes

The earlier Gaussian Naive Bayes models were trained on scaled continuous variables. While this is the standard setup for GaussianNB, the underlying credit-risk features did not align well with the model’s assumptions. Several variables were highly skewed, zero-inflated, count-based, or strongly correlated. As a result, GaussianNB was forced to approximate many non-normal feature distributions and combine overlapping signals under a conditional independence assumption.

The Bernoulli Naive Bayes model used a different feature representation: one-hot encoded binned variables. This transformed each predictor into binary risk indicators such as whether utilization fell into a specific bucket or whether delinquency counts exceeded a threshold. That representation was much more compatible with BernoulliNB, which is designed for binary presence/absence features.

This change materially improved performance because binning converted nonlinear credit relationships into explicit risk states, while one-hot encoding removed the need for Gaussian distribution assumptions. Instead of estimating probabilities from imperfect continuous shapes, the model simply learned how often each risk bucket appeared among good and bad accounts.

The performance jump demonstrates an important modeling principle: algorithm success often depends as much on feature representation as on the algorithm itself. The same Naive Bayes family that underperformed on scaled continuous inputs became highly competitive once the data structure matched the model’s assumptions.

### Practical Takeaway

- GaussianNB on scaled variables tested continuous probability assumptions.
- BernoulliNB on one-hot bins tested binary risk-state assumptions.
- The binary representation proved substantially stronger for this dataset.

### Broader Lesson

Model selection should evaluate both algorithms and data representations. In this case, changing the feature encoding unlocked far more value than changing the model family itself.

In [25]:
# Bernoulli Naive Bayes Grid

alpha_grid = [0.01, 0.05, 0.10, 0.25, 0.50, 1, 2, 5, 10]
fit_prior_grid = [True, False]

start_num = get_next_model_number(bnb_registry, "BNB")

counter = start_num

for alpha in alpha_grid:
    for fit_prior in fit_prior_grid:

        model_number = f"BNB{counter:03d}"

        model = BernoulliNB(
            alpha=alpha,
            fit_prior=fit_prior,
            binarize=None
        )

        bnb_registry = run_nb_model(
            model_registry=bnb_registry,
            model=model,
            model_number=model_number,
            model_name=f"BernoulliNB alpha={alpha}, fit_prior={fit_prior}",
            X_train=X_train,
            X_val=X_val,
            y_train=y_train,
            y_val=y_val,
            features=candidate_features_bnb,
            analyst_comments="BernoulliNB tuning grid.",
            run_permutation=False,
            display_outputs=False
        )

        counter += 1

print("BernoulliNB grid complete.")

BernoulliNB grid complete.


In [26]:
# Bernoulli Naive Bayes Summary Table

bnb_summary_rows = []

for model_num, model_data in bnb_registry.items():

    meta = model_data["metadata"]
    diag = model_data["diagnostics"]

    train_row = diag.loc[diag["dataset"] == "train"].iloc[0]
    val_row   = diag.loc[diag["dataset"] == "validation"].iloc[0]

    params = meta.get("parameters", {})

    bnb_summary_rows.append({
        "model_number": model_num,
        "model_name": meta["model_name"],
        "alpha": params.get("alpha", None),
        "fit_prior": params.get("fit_prior", None),
        "num_features": meta.get("num_features", None),

        "train_auc": train_row["auc"],
        "val_auc": val_row["auc"],
        "auc_gap": train_row["auc"] - val_row["auc"],

        "train_ks": train_row["ks"],
        "val_ks": val_row["ks"],
        "ks_gap": train_row["ks"] - val_row["ks"],

        "val_pr_auc": val_row["pr_auc"],
        "val_brier": val_row["brier_score"],
        "val_log_loss": val_row["log_loss"],

        "val_accuracy": val_row["accuracy"],
        "val_precision": val_row["precision"],
        "val_recall": val_row["recall"],
        "val_f1": val_row["f1"]
    })

bnb_summary_df = (
    pd.DataFrame(bnb_summary_rows)
    .sort_values(
        by=["val_auc", "val_ks", "val_pr_auc"],
        ascending=False
    )
    .reset_index(drop=True)
)

display(bnb_summary_df)

,model_number,model_name,alpha,fit_prior,num_features,train_auc,val_auc,auc_gap,train_ks,val_ks,ks_gap,val_pr_auc,val_brier,val_log_loss,val_accuracy,val_precision,val_recall,val_f1
0,BNB018,"BernoulliNB alpha=10, fit_prior=True",10.00,True,41,0.849884,0.855575,-0.005691,0.543363,0.558366,-0.015003,0.378278,0.075345,0.342290,0.909923,0.374636,0.514486,0.433563
1,BNB019,"BernoulliNB alpha=10, fit_prior=False",10.00,False,41,0.849884,0.855575,-0.005691,0.543363,0.558366,-0.015003,0.378278,0.132570,0.585592,0.831381,0.240838,0.704629,0.358979
2,BNB016,"BernoulliNB alpha=5, fit_prior=True",5.00,True,41,0.849812,0.855524,-0.005712,0.543463,0.558150,-0.014688,0.378190,0.075421,0.342750,0.909900,0.374485,0.514153,0.433343
3,BNB017,"BernoulliNB alpha=5, fit_prior=False",5.00,False,41,0.849812,0.855524,-0.005712,0.543463,0.558150,-0.014688,0.378190,0.132845,0.586751,0.831515,0.241002,0.704629,0.359162
4,BNB014,"BernoulliNB alpha=2, fit_prior=True",2.00,True,41,0.849797,0.855501,-0.005704,0.543410,0.558027,-0.014617,0.378125,0.075467,0.343028,0.909789,0.374031,0.514153,0.433039
5,BNB015,"BernoulliNB alpha=2, fit_prior=False",2.00,False,41,0.849797,0.855501,-0.005704,0.543410,0.558027,-0.014617,0.378125,0.133010,0.587445,0.831671,0.241195,0.704629,0.359375
6,BNB001,Bernoulli Naive Bayes - All Binned Features,1.00,True,41,0.849783,0.855490,-0.005708,0.543040,0.558075,-0.015035,0.378108,0.075482,0.343121,0.909789,0.374031,0.514153,0.433039
7,BNB012,"BernoulliNB alpha=1, fit_prior=True",1.00,True,41,0.849783,0.855490,-0.005708,0.543040,0.558075,-0.015035,0.378108,0.075482,0.343121,0.909789,0.374031,0.514153,0.433039
8,BNB013,"BernoulliNB alpha=1, fit_prior=False",1.00,False,41,0.849783,0.855490,-0.005708,0.543040,0.558075,-0.015035,0.378108,0.133064,0.587676,0.831671,0.241195,0.704629,0.359375
9,BNB010,"BernoulliNB alpha=0.5, fit_prior=True",0.50,True,41,0.849777,0.855483,-0.005706,0.543160,0.558244,-0.015084,0.378108,0.075490,0.343168,0.909789,0.374031,0.514153,0.433039


In [27]:
# Save Bernoulli Naive Bayes Champion Model Artifact

save_model_artifact(
    model_registry=bnb_registry,
    model_id="BNB018",
    model_dir=MODEL_DIR,
    config_dir=CONFIG_DIR
)

In [28]:
# Export Bernoulli Naive Bayes Champion Summary

bnb_champion_id = "BNB018"

bnb_champion_path = OUTPUT_DIR / "04c2_nb_binned_bernoulli_champion_summary.xlsx"

with pd.ExcelWriter(bnb_champion_path, engine="openpyxl") as writer:
    bnb_summary_df.to_excel(
        writer,
        sheet_name="BNB_Model_Comparison",
        index=False
    )

    bnb_registry[bnb_champion_id]["diagnostics"].to_excel(
        writer,
        sheet_name="BNB018_Diagnostics",
        index=False
    )

    bnb_registry[bnb_champion_id]["permutation_importance"].to_excel(
        writer,
        sheet_name="BNB018_Permutation",
        index=False
    )

print("Saved:", bnb_champion_path)

Saved: /Users/sumanchattopadhyay/Documents/Documents - Suman’s MacBook Air/Projects/credit-risk-mlops-lab/outputs/04c2_nb_binned_bernoulli_champion_summary.xlsx


## Bernoulli Naive Bayes Summary

Bernoulli Naive Bayes was tested on one-hot encoded binned features to align the input representation with the model’s binary feature assumptions.

This representation materially outperformed Gaussian Naive Bayes on scaled continuous variables, demonstrating that Naive Bayes performance depends heavily on feature encoding. Binned one-hot indicators allowed the model to learn risk-state likelihoods directly, such as whether a borrower fell into a high-utilization or high-delinquency bucket.

A smoothing and prior grid was tested across multiple alpha values and prior settings. Model performance was highly stable across alpha values, indicating that the main performance gain came from binning and binary feature representation rather than hyperparameter tuning.

Models with `fit_prior=True` were preferred because they preserved the observed class imbalance and produced substantially better Brier Score and Log Loss. Models with `fit_prior=False` had similar ranking metrics but poorer probability calibration.

The final Bernoulli Naive Bayes champion was **BNB018**, selected for the best validation AUC and strong calibration relative to the alternative prior settings.

**Final Bernoulli NB champion:** BNB018  
**Key takeaway:** Feature representation mattered more than hyperparameter tuning.